<a href="https://colab.research.google.com/github/charveeee/PPG-Signal-Processor/blob/main/PPG_Signal_Processor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import time
import json
import pandas as pd
import numpy as np
import scipy.signal as signal
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Ensure data folders exist
os.makedirs('data', exist_ok=True)
os.makedirs('test_data', exist_ok=True)

# 1D Convolutional U-Net Architecture
class SignalDenoisingUNet1D(nn.Module):
    def __init__(self):
        super(SignalDenoisingUNet1D, self).__init__()
        # Kernel size=9 captures full pulse geometry (~0.8s cardiac window at 50Hz)
        self.enc1 = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=9, stride=2, padding=4),
            nn.BatchNorm1d(16),
            nn.ReLU()
        )
        self.enc2 = nn.Sequential(
            nn.Conv1d(16, 32, kernel_size=9, stride=2, padding=4),
            nn.BatchNorm1d(32),
            nn.ReLU()
        )
        
        # Decoder (Upsampling)
        self.dec2 = nn.Sequential(
            nn.ConvTranspose1d(32, 16, kernel_size=9, stride=2, padding=4, output_padding=1),
            nn.BatchNorm1d(16),
            nn.ReLU()
        )
        # Unbounded linear output (no Sigmoid) to prevent systolic peak compression
        self.dec1 = nn.ConvTranspose1d(32, 1, kernel_size=9, stride=2, padding=4, output_padding=1)

    def forward(self, x):
        e1 = self.enc1(x)       # [Batch, 16, 100]
        e2 = self.enc2(e1)      # [Batch, 32, 50]
        d2 = self.dec2(e2)      # [Batch, 16, 100]
        cat1 = torch.cat((d2, e1), dim=1)  # Skip Connection [Batch, 32, 100]
        return self.dec1(cat1)

print("✅ Cell 1: Updated 1D U-Net architecture initialized.")

✅ Cell 1: Updated 1D U-Net architecture initialized.


In [2]:
def butter_bandpass_filter(data, lowcut=0.5, highcut=4.0, fs=50.0, order=2):
    nyq = 0.5 * fs
    low, high = lowcut / nyq, highcut / nyq
    b, a = signal.butter(order, [low, high], btype='band')
    return signal.filtfilt(b, a, data)

class RealWorldPPGDataset(Dataset):
    def __init__(self, csv_filepaths, window_length=200, raw_fs=500.0, target_fs=50.0):
        self.samples_noisy = []
        self.samples_clean = []
        self.window_length = window_length
        
        for file in csv_filepaths:
            if not os.path.exists(file):
                print(f"Warning: File {file} not found. Skipping...")
                continue
                
            df = pd.read_csv(file)
            
            # Identify PPG signal column
            pleth_col = None
            if 'pleth_1' in df.columns:
                pleth_col = 'pleth_1'
            else:
                cols = [c for c in df.columns if 'pleth' in c.lower()]
                if cols:
                    pleth_col = cols[0]
            
            if pleth_col is None:
                continue

            raw_sig = df[pleth_col].values.astype(np.float32)
            raw_sig = raw_sig[~np.isnan(raw_sig)]
            
            if len(raw_sig) < 100:
                continue
            
            # Resample native 500 Hz stream down to 50 Hz target
            num_samples_target = int(len(raw_sig) * (target_fs / raw_fs))
            resampled_sig = signal.resample(raw_sig, num_samples_target)
            
            # Key step: Extract pristine bandpassed baseline target (0.5 - 4.0 Hz)
            clean_reference = butter_bandpass_filter(resampled_sig, lowcut=0.5, highcut=4.0, fs=target_fs)
            
            step_size = window_length // 2
            t = np.linspace(0, 4.0, window_length)
            
            for start in range(0, len(resampled_sig) - window_length, step_size):
                clean_chunk = clean_reference[start : start + window_length]
                
                # Verify ground truth segment is non-zero
                if np.std(clean_chunk) < 1e-4:
                    continue
                
                # Synthetic in-band and out-of-band motion artifacts added in RAM
                motion_noise = 0.8 * np.sin(2 * np.pi * 0.3 * t) + np.random.normal(0, 0.15, window_length)
                noisy_chunk = clean_chunk + motion_noise
                
                # Z-Score Normalization per window
                noisy_norm = (noisy_chunk - np.mean(noisy_chunk)) / (np.std(noisy_chunk) + 1e-6)
                clean_norm = (clean_chunk - np.mean(clean_chunk)) / (np.std(clean_chunk) + 1e-6)
                
                self.samples_noisy.append(noisy_norm)
                self.samples_clean.append(clean_norm)
                
        self.samples_noisy = np.array(self.samples_noisy)
        self.samples_clean = np.array(self.samples_clean)

    def __len__(self):
        return len(self.samples_noisy)

    def __getitem__(self, idx):
        return torch.FloatTensor(self.samples_noisy[idx]).unsqueeze(0), torch.FloatTensor(self.samples_clean[idx]).unsqueeze(0)

print("✅ Cell 2: Fail-Safe RealWorldPPGDataset class loaded.")

✅ Cell 2: Fail-Safe RealWorldPPGDataset class loaded.


In [3]:
def calculate_sqi(sig, fs=50.0):
    freqs, psd = signal.welch(sig, fs=fs, nperseg=len(sig))
    cardiac_band_power = np.sum(psd[(freqs >= 0.5) & (freqs <= 4.0)])
    total_power = np.sum(psd) + 1e-8
    sqi = (cardiac_band_power / total_power) * 100.0
    return np.clip(sqi, 0.0, 100.0)

print("✅ Cell 3: SQI calculation engine active.")

✅ Cell 3: SQI calculation engine active.


In [5]:
# Cell 4: Realistic Synthetic Test Data Generator (Saved in 'test_data/')
import os
import time
import pandas as pd
import numpy as np
import scipy.signal as signal
from datetime import datetime, timedelta

os.makedirs('test_data', exist_ok=True)

def butter_bandpass_filter(data, lowcut=0.5, highcut=4.0, fs=50.0, order=2):
    nyq = 0.5 * fs
    low, high = lowcut / nyq, highcut / nyq
    b, a = signal.butter(order, [low, high], btype='band')
    return signal.filtfilt(b, a, data)

# 1. High-frequency baseline parameters (6 seconds at 500 Hz)
num_samples = 3000
fs = 500.0
t = np.linspace(0, num_samples / fs, num_samples)

# 2. Morphologically accurate dual-Gaussian PPG pulse template (~72 BPM)
hr_hz = 1.2
pulse_period = 1.0 / hr_hz

def ppg_pulse_template(p):
    systolic = np.exp(-((p - 0.25) ** 2) / 0.008)
    diastolic = 0.35 * np.exp(-((p - 0.45) ** 2) / 0.006)
    return systolic + diastolic

phase = (t % pulse_period) / pulse_period
clean_cardiac = np.array([ppg_pulse_template(p) for p in phase])

# 3. Add heavy motion artifacts & baseline wander (To evaluate model resilience)
baseline_wander = 0.4 * np.sin(2 * np.pi * 0.15 * t)
motion_artifact = 0.8 * np.sin(2 * np.pi * 0.35 * t + 0.5)
high_freq_noise = np.random.normal(0, 0.12, num_samples)

noisy_cardiac = clean_cardiac + baseline_wander + motion_artifact + high_freq_noise

# Scale to realistic ADC integer counts (~76,000 baseline)
pleth_noisy = (76000 + noisy_cardiac * 1200).astype(int)
pleth_clean = (76000 + clean_cardiac * 1200).astype(int)

# 4. Generate millisecond timestamps
start_time = datetime(2021, 1, 1, 11, 22, 57, 309233)
timestamps = [start_time + timedelta(milliseconds=i*2) for i in range(num_samples)]

# 5. Build CSV structure
df = pd.DataFrame({
    "time": [ts.strftime("%Y-%m-%d %H:%M:%S.%f") for ts in timestamps],
    "ecg": (45000 + (np.sin(2 * np.pi * 1.2 * t) * 20000) + np.random.normal(0, 800, num_samples)).astype(int),
    "pleth_1": pleth_noisy,
    "pleth_1_clean": pleth_clean,  # Pristine ground truth reference target
    "pleth_2": pleth_noisy - 5300 + np.random.randint(-20, 20, num_samples),
    "pleth_3": (3430 + noisy_cardiac * 50).astype(int),
    "a_x": np.round(9.21 + motion_artifact * 0.15, 6),
    "a_y": np.round(0.78 + motion_artifact * 0.05, 6),
    "a_z": np.round(1.97 - motion_artifact * 0.08, 6)
})

file_path = os.path.join("test_data", f"noisy_ppg_test_{int(time.time())}.csv")
df.to_csv(file_path, index=False)
print(f"✅ Cell 4: Test synthetic CSV saved to: {file_path}")

✅ Cell 4: Test synthetic CSV saved to: test_data\noisy_ppg_test_1786778810.csv


In [6]:
# Hybrid Loss combining MSE (Time Domain) and FFT L1 Loss (Frequency Domain)
class HybridSpectralLoss(nn.Module):
    def __init__(self, alpha=0.3):
        super(HybridSpectralLoss, self).__init__()
        self.mse = nn.MSELoss()
        self.alpha = alpha

    def forward(self, pred, target):
        time_loss = self.mse(pred, target)
        
        # Real FFT spectral loss forces exact cardiac band matching
        fft_pred = torch.abs(torch.fft.rfft(pred, dim=-1))
        fft_target = torch.abs(torch.fft.rfft(target, dim=-1))
        freq_loss = torch.mean(torch.abs(fft_pred - fft_target))
        
        return time_loss + (self.alpha * freq_loss)

data_folder = 'data' 
training_files = [os.path.join(data_folder, f) for f in os.listdir(data_folder) if f.endswith('.csv')]

print(f"Loading {len(training_files)} file(s) from '{data_folder}/' folder...")
dataset = RealWorldPPGDataset(csv_filepaths=training_files, window_length=200, raw_fs=500.0, target_fs=50.0)

if len(dataset) > 0:
    dataloader = DataLoader(dataset, batch_size=16, shuffle=True)
    model = SignalDenoisingUNet1D()
    criterion = HybridSpectralLoss(alpha=0.3)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    print(f"🚀 Training 1D U-Net on {len(dataset)} PPG signal windows...")
    model.train()
    num_epochs = 50

    for epoch in range(num_epochs):
        epoch_loss = 0.0
        for noisy_batch, clean_batch in dataloader:
            optimizer.zero_grad()
            outputs = model(noisy_batch)
            loss = criterion(outputs, clean_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            
        avg_loss = epoch_loss / len(dataloader)
        scheduler.step(avg_loss)
        
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] — Loss: {avg_loss:.6f}")

    torch.save(model.state_dict(), 'unet_realdata_weights.pth')
    print("✅ Cell 5: Training complete! Updated weights saved to unet_realdata_weights.pth!")
else:
    print("⚠️ No CSV files found in the 'data/' folder. Please add training files to proceed.")

Loading 67 file(s) from 'data/' folder...
🚀 Training 1D U-Net on 16113 PPG signal windows...
Epoch [1/50] — Loss: 0.187755
Epoch [10/50] — Loss: 0.026895
Epoch [20/50] — Loss: 0.020855
Epoch [30/50] — Loss: 0.018313
Epoch [40/50] — Loss: 0.016525
Epoch [50/50] — Loss: 0.015149
✅ Cell 5: Training complete! Updated weights saved to unet_realdata_weights.pth!


In [7]:
FS = 50.0
length = 200
t = np.linspace(0, 4.0, length)

def test_interactive_pipeline(window_index):
    if len(dataset) == 0:
        print("Cannot run visualization without loaded dataset samples.")
        return
        
    window_idx = min(window_index, len(dataset) - 1)
    noisy_tensor, clean_tensor = dataset[window_idx]
    
    raw_noisy = noisy_tensor.squeeze().numpy()
    true_clean = clean_tensor.squeeze().numpy()
    
    filtered_signal = butter_bandpass_filter(raw_noisy, lowcut=0.5, highcut=4.0, fs=FS)
    filtered_signal = (filtered_signal - np.mean(filtered_signal)) / (np.std(filtered_signal) + 1e-8)
    
    model.eval()
    with torch.no_grad():
        input_sample = noisy_tensor.unsqueeze(0)
        reconstructed = model(input_sample).squeeze().numpy()
        
    sqi_score = calculate_sqi(reconstructed, fs=FS)
    min_distance = max(1, int(FS * 0.35))
    peaks, _ = signal.find_peaks(reconstructed, distance=min_distance, prominence=0.15)
    
    if len(peaks) >= 2:
        est_bpm = 60.0 / np.mean(np.diff(t[peaks]))
        bpm_str = f"{est_bpm:.1f} BPM"
    else:
        bpm_str = "N/A"

    print(f"Window: {window_idx} | Reconstructed SQI: {sqi_score:.1f}% | Est. Heart Rate: {bpm_str} | Peaks: {len(peaks)}")
    
    plt.figure(figsize=(12, 8))
    
    plt.subplot(3, 1, 1)
    plt.plot(t, raw_noisy, color='crimson', label='Raw Input Window')
    plt.title(f'Stage 1: Raw Input Stream (Window #{window_idx})')
    plt.legend(loc='upper right')
    
    plt.subplot(3, 1, 2)
    plt.plot(t, filtered_signal, color='darkorange', label='Classical Bandpass Filter')
    plt.title('Stage 2: Butterworth Bandpass Filter (0.5 - 4.0 Hz)')
    plt.legend(loc='upper right')
    
    plt.subplot(3, 1, 3)
    plt.plot(t, reconstructed, color='royalblue', linewidth=2, label='1D U-Net Reconstruction')
    if len(peaks) > 0:
        plt.scatter(t[peaks], reconstructed[peaks], color='purple', s=80, zorder=5, label=f'Detected Peaks ({len(peaks)})')
    plt.title(f'Stage 3: 1D U-Net Denoised Output (SQI Score: {sqi_score:.1f}%)')
    plt.legend(loc='upper right')
    
    plt.tight_layout()
    plt.show()

if len(dataset) > 0:
    interact(
        test_interactive_pipeline,
        window_index=IntSlider(min=0, max=max(0, len(dataset) - 1), step=1, value=0, description='Window Index')
    );

interactive(children=(IntSlider(value=0, description='Window Index', max=16112), Output()), _dom_classes=('wid…